# Transformer Notebook 01: Attention Mechanisms & Heatmaps

This notebook provides interactive Python implementations and visual heatmaps for **Scaled Dot-Product Attention** and **Multi-Head Attention (MHA)** head diversity.

### Core Concepts Covered:
1. **Scaled Dot-Product Attention**: Why dividing by $\sqrt{d_k}$ prevents softmax saturation.
2. **Attention Weight Visualizations**: Plotting full attention heatmaps across multiple sequence tokens.
3. **Multi-Head Attention (MHA)**: Simulating how different attention heads specialize in distinct syntactic and semantic relationships.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set dark aesthetic style
plt.style.use('dark_background')
np.random.seed(42)

def scaled_dot_product_attention(Q, K, V, scale=True):
    d_k = Q.shape[-1]
    scores = np.dot(Q, K.T)
    if scale:
        scores = scores / np.sqrt(d_k)
    
    # Softmax calculation
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    output = np.dot(weights, V)
    return output, weights

print('Attention helper function defined successfully!')

## Part 1: Scaled vs. Unscaled Attention Softmax Saturation

Let me demonstrate mathematically what happens when $d_k$ grows large (e.g. $d_k = 128$). Without scaling by $\sqrt{d_k}$, raw dot products grow in variance, pushing softmax probabilities to $0.0$ and $1.0$.

In [ ]:
seq_len, d_k = 10, 128
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

_, unscaled_weights = scaled_dot_product_attention(Q, K, V, scale=False)
_, scaled_weights = scaled_dot_product_attention(Q, K, V, scale=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
im1 = ax1.imshow(unscaled_weights, cmap='magma', vmin=0, vmax=1)
ax1.set_title(f'Unscaled Attention (d_k={d_k})\nSoftmax Saturated', color='#ff6b6b')
fig.colorbar(im1, ax=ax1, shrink=0.8)

im2 = ax2.imshow(scaled_weights, cmap='viridis', vmin=0, vmax=0.5)
ax2.set_title(f'Scaled Attention (d_k={d_k}, factor=1/sqrt({d_k}))\nActive Softmax', color='#51cf66')
fig.colorbar(im2, ax=ax2, shrink=0.8)

plt.tight_layout()
plt.show()

## Part 2: Visualizing Multi-Head Attention Diversity

Different attention heads learn different linguistic patterns. Let's visualize four synthetic head patterns over a sample sentence.

In [ ]:
tokens = ['The', 'animal', 'didn\'t', 'cross', 'street', 'because', 'it', 'was', 'tired']
N = len(tokens)

# Simulate head distributions
h1 = np.eye(N, k=1) * 0.7 + np.eye(N) * 0.3 # Local next-token
h2 = np.zeros((N, N))
h2[6, 1] = 0.85; h2[6, 6] = 0.15 # Coreference ('it' -> 'animal')
for i in range(N):
    if i != 6: h2[i, i] = 1.0

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(h1, cmap='cool')
axes[0].set_title('Head 1: Syntactic Local Next-Token Adjacency')
axes[0].set_xticks(range(N)); axes[0].set_yticks(range(N))
axes[0].set_xticklabels(tokens, rotation=45); axes[0].set_yticklabels(tokens)

axes[1].imshow(h2, cmap='spring')
axes[1].set_title('Head 2: Coreference Resolution (it -> animal)')
axes[1].set_xticks(range(N)); axes[1].set_yticks(range(N))
axes[1].set_xticklabels(tokens, rotation=45); axes[1].set_yticklabels(tokens)

plt.tight_layout()
plt.show()